In [1]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

model_path = r'mp_model/pose_landmarker_full.task'

import mediapipe as mp

BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

# Create a hand landmarker instance with the video mode:
options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.VIDEO,
    num_poses = 4,
    output_segmentation_masks=True)
# detector = vision.HandLandmarker.create_from_options(options)

In [2]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision import drawing_utils
from mediapipe.tasks.python.vision import drawing_styles
import numpy as np
import matplotlib.pyplot as plt


def draw_landmarks_on_image(rgb_image, detection_result):
  pose_landmarks_list = detection_result.pose_landmarks
  annotated_image = np.copy(rgb_image)

  pose_landmark_style = drawing_styles.get_default_pose_landmarks_style()
  pose_connection_style = drawing_utils.DrawingSpec(color=(0, 255, 0), thickness=2)

  for pose_landmarks in pose_landmarks_list:
    drawing_utils.draw_landmarks(
        image=annotated_image,
        landmark_list=pose_landmarks,
        connections=vision.PoseLandmarksConnections.POSE_LANDMARKS,
        landmark_drawing_spec=pose_landmark_style,
        connection_drawing_spec=pose_connection_style)

  return annotated_image

In [14]:
import cv2
import time
import numpy as np
import mediapipe as mp

def draw_segmentation_on_image(rgb_image, detection_result, alpha=0.5):
    annotated_image = np.copy(rgb_image)

    if not detection_result.segmentation_masks:
        return annotated_image

    segmentation_mask = detection_result.segmentation_masks[0].numpy_view()
    segmentation_mask = np.squeeze(segmentation_mask)

    # 파란색 오버레이
    overlay = np.zeros_like(rgb_image, dtype=np.uint8)
    overlay[:, :, 2] = (segmentation_mask * 255).astype(np.uint8)

    annotated_image = cv2.addWeighted(
        annotated_image, 1.0,
        overlay, alpha,
        0
    )
    return annotated_image


with PoseLandmarker.create_from_options(options) as landmarker:
    cap = cv2.VideoCapture(0)

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # OpenCV frame(BGR) -> RGB
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # 데이터 준비
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        timestamp = int(time.time() * 1000)
        pose_landmarker_result = landmarker.detect_for_video(mp_image, timestamp)

        # segmentation 먼저
        annotated_image = draw_segmentation_on_image(rgb_frame, pose_landmarker_result, alpha=0.9)

        # 그 위에 포즈 랜드마크
        annotated_image = draw_landmarks_on_image(annotated_image, pose_landmarker_result)

        # 화면 출력은 다시 BGR로
        cv2.imshow("video", cv2.cvtColor(annotated_image, cv2.COLOR_RGB2BGR))

        key = cv2.waitKey(1)
        if key == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

In [3]:
import cv2
import time
import mediapipe as mp
import numpy as np
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision import drawing_utils, drawing_styles

BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
FaceLandmarker = mp.tasks.vision.FaceLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
RunningMode = mp.tasks.vision.RunningMode

pose_options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=r"mp_model/pose_landmarker_full.task"),
    running_mode=RunningMode.VIDEO,
    num_poses=1
)

face_options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=r"mp_model/face_landmarker.task"),
    running_mode=RunningMode.VIDEO,
    num_faces=1
)

def draw_pose(rgb_image, result):
    annotated = np.copy(rgb_image)
    for pose_landmarks in result.pose_landmarks:
        drawing_utils.draw_landmarks(
            image=annotated,
            landmark_list=pose_landmarks,
            connections=vision.PoseLandmarksConnections.POSE_LANDMARKS,
            landmark_drawing_spec=drawing_styles.get_default_pose_landmarks_style(),
            connection_drawing_spec=drawing_utils.DrawingSpec(color=(0, 255, 0), thickness=2),
        )
    return annotated

def draw_face(rgb_image, result):
    annotated = np.copy(rgb_image)
    for face_landmarks in result.face_landmarks:
        drawing_utils.draw_landmarks(
            image=annotated,
            landmark_list=face_landmarks,
            connections=vision.FaceLandmarksConnections.FACE_LANDMARKS_TESSELATION,
            landmark_drawing_spec=None,
            connection_drawing_spec=drawing_styles.get_default_face_mesh_tesselation_style(),
        )
        drawing_utils.draw_landmarks(
            image=annotated,
            landmark_list=face_landmarks,
            connections=vision.FaceLandmarksConnections.FACE_LANDMARKS_CONTOURS,
            landmark_drawing_spec=None,
            connection_drawing_spec=drawing_styles.get_default_face_mesh_contours_style(),
        )
        drawing_utils.draw_landmarks(
            image=annotated,
            landmark_list=face_landmarks,
            connections=vision.FaceLandmarksConnections.FACE_LANDMARKS_LEFT_IRIS,
            landmark_drawing_spec=None,
            connection_drawing_spec=drawing_styles.get_default_face_mesh_iris_connections_style(),
        )
        drawing_utils.draw_landmarks(
            image=annotated,
            landmark_list=face_landmarks,
            connections=vision.FaceLandmarksConnections.FACE_LANDMARKS_RIGHT_IRIS,
            landmark_drawing_spec=None,
            connection_drawing_spec=drawing_styles.get_default_face_mesh_iris_connections_style(),
        )
    return annotated

with PoseLandmarker.create_from_options(pose_options) as pose_landmarker, \
     FaceLandmarker.create_from_options(face_options) as face_landmarker:

    cap = cv2.VideoCapture(0)

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        ts = int(time.time() * 1000)

        pose_result = pose_landmarker.detect_for_video(mp_image, ts)
        face_result = face_landmarker.detect_for_video(mp_image, ts)

        annotated = draw_pose(rgb, pose_result)
        annotated = draw_face(annotated, face_result)

        cv2.imshow("video", cv2.cvtColor(annotated, cv2.COLOR_RGB2BGR))

        if cv2.waitKey(1) == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()